# Chapter 10: Tool Engineering
## Topic 3: Designing `validate_fd_reference` Properly

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topics 1 and 2 established the general architecture of tool-calling — a schema the model reads, a real function the model never sees, a strict message-passing convention. This topic takes that general architecture and asks a much more specific, practical question: what does it actually take to design *one particular tool* — this project's own `validate_fd_reference` — so it behaves correctly and safely across every real-world case it will actually encounter, not just the easy, expected one?
- The core design principle this topic centers on, already present in this project's real implementation: **a tool that performs a real lookup must be able to distinguish every meaningfully different outcome, not just success versus generic failure.** "The reference number doesn't exist at all" and "the reference number exists but something is unusual about the record" are genuinely different situations that call for different downstream behavior — collapsing them into a single boolean or a single error would destroy information the calling agent (and ultimately, the human it may need to escalate to) genuinely needs.
- This connects directly to two things already established elsewhere in this notebook series: Chapter 9 Topic 4 argued customer-record lookups must be exact-match only, never similarity-based — this topic is about designing that exact-match tool's *output* properly, once the lookup mechanism itself is already correctly chosen. And Chapter 8 Topic 3's faithfulness/relevance/correctness framework depends on tools returning information precise enough to support or contradict a specific claim — a vague or collapsed tool result undermines that entire downstream verification chain before it even gets a chance to run.


### 2. Internal Working — Step by Step

**The concrete design decisions actually present in this project's real `validate_fd_reference`, examined one at a time:**

1. **The critical `found: False` vs `found: True` distinction, and why it's the tool's most important design choice.** A reference number that simply doesn't exist in the records table at all (a typo, a fabricated number, a reference from a different institution entirely) is a fundamentally different situation from one that exists and has a real status — the docstring itself makes this explicit: *"Returns found=False if no such reference exists in the system at all — which is meaningfully different from 'exists but something's off', and the agent needs to be able to tell the two apart."* Collapsing these into one generic "invalid" result would force the calling agent (and the model reasoning over its result) to treat a simple typo identically to a genuinely suspicious or complex case.
2. **Returning multiple, individually-named fields rather than one combined summary string.** The real function returns `customer_name_on_record`, `status`, `fd_amount_inr`, and `maturity_date` as separate, individually addressable fields — not a single pre-formatted sentence like `"Found: Shobha Chopra, Closed, ₹391,000"`. This matters because it lets the calling model (or any downstream code) reference exactly the specific field it needs, rather than having to parse meaning back out of a string that was already flattened into prose.
3. **Input sanitization before the lookup runs at all.** `reference_number.strip()` handles the ordinary, expected case where a model-extracted reference number might carry incidental leading or trailing whitespace from how it was embedded in email text — a small but real design choice that prevents an otherwise-valid reference number from failing to match purely due to formatting noise, independent of whether the reference itself is genuinely valid.
4. **Echoing the input `reference_number` back in the result, even when nothing was found.** Both the `found: True` and `found: False` branches include the original `reference_number` in the returned dict — this seems minor, but it means the tool's result is self-contained and traceable on its own, without requiring the calling code to separately remember which specific input this particular result corresponds to (directly relevant once Topic 2's `tool_use_id` linkage and this chapter's upcoming multi-tool-agent topic are involved, where multiple results might need to be reasoned about together).
5. **The function's return type is a plain, structured `dict`, not a special result object or a raw database row.** This is a deliberate simplicity choice — the result needs to be serializable (Topic 2's `json.dumps(result)` when constructing the `tool_result` block) and needs to be a shape the model can straightforwardly read back and reason over as structured data, not something requiring additional parsing or transformation before it's usable.


### 3. How This Is Implemented in This Project

- `validate_fd_reference`'s real implementation delegates the actual lookup to a separate function, `get_fd_record()`, defined in `fd_database.py` — this is itself a good design separation: `validate_fd_reference` is the *tool-facing* function (matching the schema's contract, shaping the result the model will see), while `get_fd_record()` is a lower-level, reusable data-access function that doesn't know or care that it's being used inside a tool at all. This separation means `get_fd_record()` could be reused by other tools or other parts of the system without dragging along any tool-specific shaping logic.
- The underlying `fd_master_database.csv` and its `get_fd_record()` accessor represent this project's mock stand-in for what would be a real SQL database in production — the tool design principles here (found/not-found distinction, structured multi-field output, input sanitization) apply identically regardless of whether the underlying store is a CSV file or a production database, which is precisely the point of designing the tool's *interface* carefully, independent of its backing implementation.
- This tool's design connects directly to Chapter 3's system prompt instruction ("if the email contains something that looks like an FD reference number, you MUST call validate_fd_reference before deciding the label") — the tool was specifically designed to support that mandatory-call pattern, which depends on the tool reliably returning a clear, structured answer the model can act on deterministically, not an ambiguous or partial one.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **A collapsed found/not-found distinction is a subtle, easy-to-introduce bug with real downstream consequences.** If a tool were redesigned to simply return `None` or raise a generic exception for "not found," the calling agent would lose the ability to distinguish a customer's simple typo from a case that might warrant escalation or special handling — this is exactly the kind of design regression that could be introduced accidentally during a refactor without anyone noticing until a specific downstream case breaks.
- **Input sanitization needs to be scoped to what's actually expected, not applied blindly.** `.strip()` handles whitespace noise safely, but a tool accepting more complex or structured input needs to think carefully about what sanitization is safe to apply automatically versus what should instead trigger the validation-and-refuse behavior demonstrated in Topic 1's Module 3 — over-aggressive "helpful" sanitization can silently mask a genuinely malformed input that should have been caught and flagged instead.
- **Structured, multi-field output is what makes downstream faithfulness checking (Chapter 8 Topic 3/4) actually tractable.** If this tool returned one flattened summary string, extracting and checking a specific claim (like "does the amount cited match the record") would require re-parsing that string — a fragile, error-prone step this project's actual field-by-field structure avoids entirely.
- **Monitoring:** logging both the input `reference_number` and the full structured result (found status and all fields) for every real call to this tool creates exactly the audit trail Chapter 9 Topic 4 required for customer-record access — this project's existing return-shape (echoing the input, structured fields) is what makes that logging straightforward to implement without extra transformation.
- **Cost and latency:** since this is a simple, deterministic lookup against a small table (or, in production, an indexed database query), this tool's cost and latency profile is negligible compared to the retrieval-heavy `search_knowledge_base` tool (Chapter 9 Topic 3) — this asymmetry is worth remembering when reasoning about an agent loop's overall latency budget across a turn that might call several different tools.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Returning `found: False` with an echoed reference number vs raising an exception for a not-found case:** returning a structured "not found" result (this project's actual choice) lets the calling agent reason about this outcome as ordinary, expected data — a real, meaningful business outcome, not an exceptional system failure. Raising an exception instead would conflate "the customer's number doesn't exist" (a completely normal, expected occurrence) with "something went wrong in the system" (a genuine error) — these deserve different handling, and this chapter's upcoming error-handling topic (Topic 5) will draw this distinction further.
- **How many fields to return, and in what shape:** this tool returns exactly the fields a downstream classification or answer-generation step is likely to need (customer name, status, amount, maturity date) — not the entire raw database row, and not a single pre-digested summary. This is a deliberate middle ground: enough structure for the model to reason precisely, without exposing implementation details (column names, internal formatting) that aren't the tool's concern to reveal.
- **Where sanitization logic belongs — inside the tool function itself, or in the calling/dispatch layer:** this project puts `.strip()` inside `validate_fd_reference()` itself, keeping the tool self-contained and correct regardless of which calling code invokes it — a reasonable default, though Topic 1's more security-sensitive validation (format-checking against a regex before executing at all) was shown living in the dispatch layer instead. Both placements are valid; the key design principle is that *some* layer performs this work reliably, not that it must live in one specific place.


### 6. Alternatives and When to Use Each

- **Distinct, structured found/not-found result (this project's actual design):** the right choice whenever "not found" is a normal, expected, business-meaningful outcome rather than a system error — exactly this tool's situation, since customers routinely provide typo'd or incorrect reference numbers as an ordinary part of real usage.
- **Raising an exception for a not-found case:** more appropriate when "not found" genuinely represents an abnormal system state that should halt normal processing — not the right fit for a customer-facing lookup where "not found" is a routine, expected occurrence.
- **Returning the entire raw database row rather than a curated field subset:** simpler to implement, but exposes internal implementation details (column names, internal-only fields) to the model unnecessarily, and risks leaking information the model and the system prompt were never designed to expose to the ultimate customer-facing output.
- **Returning one pre-formatted summary string instead of structured fields:** convenient for direct human reading, but actively harmful for the structured verification this notebook series depends on (Chapter 8's faithfulness and citation checks) — should be avoided for any tool whose output needs to support precise, field-level downstream reasoning.


### 7. Common Mistakes and Production Failures

- Collapsing "reference number doesn't exist" and "reference number exists but something is unusual" into a single generic failure result, losing information the calling agent genuinely needs to act correctly.
- Returning a single flattened summary string instead of individually-addressable structured fields, making precise downstream faithfulness or citation checking fragile or impossible without re-parsing.
- Applying overly aggressive automatic input sanitization that silently masks a genuinely malformed input, rather than surfacing it for explicit validation and refusal.
- Not echoing the original input back in the tool's result, making a given result harder to trace back to the specific request it answers once multiple tool calls are involved in a single turn.
- Exposing raw internal database structure (column names, internal-only fields, file paths) through the tool's result, when a curated, purpose-built subset of fields would serve the model's actual needs without unnecessary exposure.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why does `validate_fd_reference` return a structured `found: False` result instead of raising an exception when a reference number doesn't exist?
  A: A customer providing an incorrect or typo'd reference number is a routine, expected business outcome, not a system error — treating it as ordinary structured data lets the calling agent reason about and handle it as a normal case (for example, asking the customer to double-check the number), rather than conflating it with a genuine system failure that an exception would typically represent.

- Q: Why does the tool return several individually-named fields rather than one combined summary string?
  A: Structured, individually-addressable fields let downstream code (and the model itself) reference exactly the specific piece of information it needs, and make precise verification — like checking whether a specific cited amount matches the record — straightforward. A single flattened string would require re-parsing to extract any specific fact, which is fragile and error-prone.

**Intermediate**

- Q: What's the significance of the docstring's explicit statement that "found=False... is meaningfully different from 'exists but something's off'"?
  A: It documents a deliberate design decision: this tool is built to distinguish between genuinely different outcomes rather than collapsing them into a single notion of "invalid." This distinction matters because the correct downstream behavior differs — a nonexistent reference number likely warrants asking the customer to recheck it, while a reference number that exists but has an unusual status might warrant a different kind of handling or escalation entirely.

- Q: Why does the tool echo the original `reference_number` back in its result, even when nothing was found?
  A: This makes the tool's result self-contained and traceable — the calling code (and, in a multi-tool scenario, potentially the model reasoning over several results at once) doesn't need to separately track which specific input a given result corresponds to, since the result carries that information along with it.

**Advanced**

- Q: A teammate proposes simplifying `validate_fd_reference` by having it return the entire raw database row directly, rather than a curated set of named fields. What would you push back on?
  A: Returning the raw row exposes internal implementation details — actual column names, internal-only fields never meant for the model or downstream consumption, and the underlying data structure's specific shape — that the tool's design should otherwise encapsulate. A curated, purpose-built field set is a deliberate interface choice: it exposes exactly what the model needs to reason correctly, and nothing more, making the tool more maintainable (the underlying database schema can change without necessarily changing the tool's contract) and more secure (nothing unintended leaks through).

- Q: How does this tool's design specifically support the faithfulness and citation verification work from Chapter 8?
  A: Faithfulness checking (Chapter 8 Topic 3/4) needs to verify specific claims against specific pieces of retrieved or looked-up information — this requires that information to be structured and individually addressable. Because `validate_fd_reference` returns distinct fields like `fd_amount_inr` and `status` rather than a single prose summary, a downstream check can directly compare "the answer claims Rs 391,000" against "the record's `fd_amount_inr` field says 391000" without needing to parse meaning back out of a flattened string — this tool's output shape is precisely what makes that kind of precise, field-level verification tractable at all.

**Scenario-based**

- Q: A production incident report shows the agent gave a customer an answer treating their genuinely nonexistent FD reference number as if it were a valid, closed account. Walk through your investigation, focusing on this tool's design.
  A: First check whether `validate_fd_reference` actually returned `found: False` correctly for this specific reference number, or whether it incorrectly matched to a different record — if the tool's own found/not-found logic is working correctly, the bug more likely lies in how the calling agent or the generation layer interpreted or handled a `found: False` result, rather than in the tool itself. This is exactly why the found/not-found distinction needs to be unambiguous and thoroughly tested at the tool level (this chapter's upcoming testing topic) — a well-designed tool result should make this kind of downstream misinterpretation harder to produce in the first place, not easier.

**Follow-up questions to expect:**

- "How would you extend this tool's design if the underlying data source changed from a CSV to a live production database?"
- "What additional fields, if any, would you consider adding to this tool's result, and how would you decide?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This tool's found/not-found distinction is a specific instance of a much more general software design principle: distinguishing "the operation succeeded and returned an empty or negative result" from "the operation failed to run correctly at all."** This distinction — often discussed generally as the difference between a legitimate negative result and an actual error — recurs constantly in API and interface design well beyond this specific tool or even LLM tool-calling broadly.
- **The separation between `validate_fd_reference` (the tool-facing shaping function) and `get_fd_record` (the lower-level, reusable data-access function) is an instance of a general layering principle in software design** — keeping a thin, purpose-specific interface layer separate from a more general, reusable underlying implementation, so that the underlying implementation can be reused elsewhere without carrying along logic specific to only one calling context.
- **This topic's principles directly generalize to Topic 6's multi-tool-agent discussion:** every additional tool added to this project's agent (`lookup_product_catalog`, `check_sender_history`) should be designed with the same care given to `validate_fd_reference` here — a clear found/not-found (or equivalent) distinction, structured multi-field output, and appropriate input handling — rather than each new tool inventing its own inconsistent conventions.

### 10. Quick Revision Summary (for last-minute interview prep)

> Designing a tool properly means more than making it technically callable — `validate_fd_reference`'s real implementation demonstrates several concrete, transferable design principles. It distinguishes "reference number doesn't exist at all" from "exists but something's unusual" as genuinely different, differently-actionable outcomes, rather than collapsing them into one generic failure. It returns individually-named, structured fields (customer name, status, amount, maturity date) instead of one flattened summary string, which is precisely what makes downstream faithfulness and citation verification (Chapter 8) tractable. It sanitizes expected input noise (whitespace) without over-reaching into masking genuinely malformed input. It echoes the original input back in its result, keeping that result self-contained and traceable. And it delegates the actual lookup to a separate, lower-level, reusable data-access function, keeping tool-specific shaping logic cleanly separated from general data access. These principles — meaningful outcome distinctions, structured output, careful input handling, traceable results, clean separation of concerns — are the concrete, applicable standard every other tool in this project (and this chapter's upcoming multi-tool-agent topic specifically) should be held to.


### Module 1: The Real Tool, Rebuilt Faithfully

Rebuild `validate_fd_reference` and its underlying `get_fd_record` exactly matching the project's actual design -- separate layers, structured output, input sanitization.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: validate_fd_reference, rebuilt faithfully to the real
# project design -- separate layers, structured output, sanitization
# ------------------------------------------------------------------

FD_RECORDS_TABLE = {
    "BJ2019FD7717": {"Customer_Name": "Shobha Chopra", "Status": "Closed (Premature)",
                      "FD_Amount_INR": 391000, "Maturity_Date": "2024-03-15"},
    "BJ2022FD6918": {"Customer_Name": "Anita Mishra", "Status": "Active",
                      "FD_Amount_INR": 160000, "Maturity_Date": "2026-11-02"},
}

def get_fd_record(reference_number: str, path: str = None) -> dict:
    """Lower-level, reusable data-access function -- knows NOTHING
    about tools, schemas, or the model. Just a plain lookup, exactly
    mirroring the real project's separation of concerns."""
    return FD_RECORDS_TABLE.get(reference_number)  # None if not found


def validate_fd_reference(reference_number: str) -> dict:
    """The REAL tool-facing function, rebuilt faithfully:
    - sanitizes expected input noise (.strip())
    - distinguishes found=False from found=True explicitly
    - echoes the input back in EVERY result
    - returns structured, individually-named fields, never a
      flattened summary string"""
    clean_ref = reference_number.strip()
    record = get_fd_record(clean_ref)

    if record is None:
        return {"reference_number": reference_number, "found": False}

    return {
        "reference_number": reference_number,
        "found": True,
        "customer_name_on_record": record["Customer_Name"],
        "status": record["Status"],
        "fd_amount_inr": record["FD_Amount_INR"],
        "maturity_date": record["Maturity_Date"],
    }


print("=" * 70)
print("MODULE 1: THE REAL TOOL, REBUILT FAITHFULLY")
print("=" * 70)

test_cases = [
    ("BJ2019FD7717", "exact match, no noise"),
    ("  BJ2022FD6918  ", "valid reference WITH whitespace noise"),
    ("BJ9999FD0000", "genuinely nonexistent reference number"),
]

for ref, label in test_cases:
    result = validate_fd_reference(ref)
    print(f"\n[{label}]")
    print(f"  Input: '{ref}'")
    print(f"  Result: {result}")

print("\nNotice: whitespace noise did NOT prevent a valid match (sanitization")
print("working correctly), and the genuinely nonexistent reference correctly")
print("returned found=False WITH the original input echoed back -- exactly")
print("the structured, traceable design the theory describes.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: THE REAL TOOL, REBUILT FAITHFULLY

[exact match, no noise]
  Input: 'BJ2019FD7717'
  Result: {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}

[valid reference WITH whitespace noise]
  Input: '  BJ2022FD6918  '
  Result: {'reference_number': '  BJ2022FD6918  ', 'found': True, 'customer_name_on_record': 'Anita Mishra', 'status': 'Active', 'fd_amount_inr': 160000, 'maturity_date': '2026-11-02'}

[genuinely nonexistent reference number]
  Input: 'BJ9999FD0000'
  Result: {'reference_number': 'BJ9999FD0000', 'found': False}

Notice: whitespace noise did NOT prevent a valid match (sanitization
working correctly), and the genuinely nonexistent reference correctly
returned found=False WITH the original input echoed back -- exactly
the structured, traceable design the theory describes.

Module 1 complete. Run Module 2 next.


### Module 2: Why the found/not-found Distinction Matters — A Concrete Failure Comparison

Build a DELIBERATELY WORSE version that collapses found/not-found into a single generic result, and show concretely what information gets lost for downstream reasoning.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Collapsed vs distinct found/not-found -- concrete comparison
# ------------------------------------------------------------------

def validate_fd_reference_BAD_DESIGN(reference_number: str) -> str:
    """A DELIBERATELY WORSE design -- collapses everything into one
    flattened string, with no way to programmatically distinguish
    'not found' from 'found but unusual' from 'found and normal'."""
    clean_ref = reference_number.strip()
    record = get_fd_record(clean_ref)
    if record is None:
        return "Could not find this FD reference."
    return f"Found: {record['Customer_Name']}, status {record['Status']}, amount {record['FD_Amount_INR']}"


print("=" * 70)
print("COLLAPSED (BAD) DESIGN vs STRUCTURED (REAL) DESIGN")
print("=" * 70)

test_ref_found = "BJ2019FD7717"
test_ref_not_found = "BJ9999FD0000"

bad_found = validate_fd_reference_BAD_DESIGN(test_ref_found)
bad_not_found = validate_fd_reference_BAD_DESIGN(test_ref_not_found)

good_found = validate_fd_reference(test_ref_found)
good_not_found = validate_fd_reference(test_ref_not_found)

print(f"\nBAD design output for FOUND case:     '{bad_found}'")
print(f"BAD design output for NOT-FOUND case: '{bad_not_found}'")

print(f"\nGOOD design output for FOUND case:     {good_found}")
print(f"GOOD design output for NOT-FOUND case: {good_not_found}")

# REAL, concrete test: can calling code programmatically check
# "was this found?" WITHOUT string-parsing, for each design?
def can_check_found_programmatically_bad(output: str) -> bool:
    """With the bad design, this requires FRAGILE STRING MATCHING --
    demonstrating exactly the problem the theory describes."""
    return "Could not find" not in output  # fragile: depends on exact wording

def can_check_found_programmatically_good(output: dict) -> bool:
    """With the good design, this is a direct, reliable field access."""
    return output["found"]

print(f"\nChecking 'was it found?' for the FOUND case:")
print(f"  BAD design requires string-matching:  {can_check_found_programmatically_bad(bad_found)}")
print(f"  GOOD design is a direct field access:  {can_check_found_programmatically_good(good_found)}")

print(f"\nWhat happens if the BAD design's wording ever changes slightly")
print(f"(e.g. 'Could not locate' instead of 'Could not find')?")
modified_bad_output = "Could not locate this FD reference."
print(f"  New wording: '{modified_bad_output}'")
print(f"  string-match check now returns: {can_check_found_programmatically_bad(modified_bad_output)}")
print(f"  (WRONG -- should be False/not-found, but the fragile string check")
print(f"   breaks silently the moment wording changes even slightly)")

print("\nThis is the CONCRETE, measurable cost of collapsing structured")
print("data into a flattened string: downstream code becomes fragile,")
print("silently breakable, and requires re-parsing meaning that was")
print("already available as real, structured data before it was flattened.")
print("\nModule 2 complete. Run Module 3 next.")


COLLAPSED (BAD) DESIGN vs STRUCTURED (REAL) DESIGN

BAD design output for FOUND case:     'Found: Shobha Chopra, status Closed (Premature), amount 391000'
BAD design output for NOT-FOUND case: 'Could not find this FD reference.'

GOOD design output for FOUND case:     {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}
GOOD design output for NOT-FOUND case: {'reference_number': 'BJ9999FD0000', 'found': False}

Checking 'was it found?' for the FOUND case:
  BAD design requires string-matching:  True
  GOOD design is a direct field access:  True

What happens if the BAD design's wording ever changes slightly
(e.g. 'Could not locate' instead of 'Could not find')?
  New wording: 'Could not locate this FD reference.'
  string-match check now returns: True
  (WRONG -- should be False/not-found, but the fragile string check
   breaks silently the moment wording c

### Module 3: Traceability — Echoing the Input Back, Tested

Demonstrates concretely why echoing the input matters, using a multi-call scenario where results need to be matched back to their originating requests.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Traceability -- echoing the input back, tested concretely
# ------------------------------------------------------------------

# simulate a scenario with SEVERAL tool results arriving together
# (as would happen in a multi-tool-call turn, Topic 6's territory)
multiple_requests = ["BJ2019FD7717", "BJ9999FD0000", "BJ2022FD6918"]
results = [validate_fd_reference(ref) for ref in multiple_requests]

print("=" * 70)
print("TRACEABILITY: matching results back to their requests")
print("=" * 70)
print("Multiple results arrived together (simulating a multi-tool turn):\n")
for r in results:
    print(f"  {r}")

# REAL test: can we correctly match each result back to its request,
# using ONLY the result itself (no external tracking needed)?
print("\nMatching each result back to its original request, using ONLY")
print("the echoed 'reference_number' field inside each result itself:")
for original_ref in multiple_requests:
    matching_result = next(r for r in results if r["reference_number"] == original_ref)
    found_status = matching_result["found"]
    print(f"  Request '{original_ref}' -> matched result: found={found_status}")

print("\nThis worked because EVERY result carries its own originating input")
print("along with it -- no separate bookkeeping needed by the calling code")
print("to remember 'which result answers which request'. This becomes")
print("essential once Topic 6's multi-tool agents call several different")
print("tools, potentially with several instances of the SAME tool, in a")
print("single turn.")

print("\nModule 3 complete. All Chapter 10 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  found=False vs found=True is a DELIBERATE, meaningful distinction --
  never collapse "doesn't exist" and "exists but unusual" into one
  generic failure state.

  Structured, individually-named fields (not a flattened summary
  string) are what make downstream faithfulness/citation checking
  (Chapter 8) tractable -- demonstrated concretely: string-matching a
  flattened output is fragile and breaks silently on wording changes.

  Sanitize EXPECTED noise (whitespace) inside the tool; don't over-reach
  into masking genuinely malformed input that should be validated and
  refused instead (Topic 1's validation-before-execution principle).

  ALWAYS echo the original input back in the result -- this is what
  makes results self-contained and traceable, critical once multiple
  tool calls are in flight in a single turn (Topic 6).

  Keep tool-facing shaping logic (validate_fd_reference) SEPARATE from
  general, reusable data access (get_fd_record) -- a clean layering
  principle that generalizes to every other tool in this project.
""")


TRACEABILITY: matching results back to their requests
Multiple results arrived together (simulating a multi-tool turn):

  {'reference_number': 'BJ2019FD7717', 'found': True, 'customer_name_on_record': 'Shobha Chopra', 'status': 'Closed (Premature)', 'fd_amount_inr': 391000, 'maturity_date': '2024-03-15'}
  {'reference_number': 'BJ9999FD0000', 'found': False}
  {'reference_number': 'BJ2022FD6918', 'found': True, 'customer_name_on_record': 'Anita Mishra', 'status': 'Active', 'fd_amount_inr': 160000, 'maturity_date': '2026-11-02'}

Matching each result back to its original request, using ONLY
the echoed 'reference_number' field inside each result itself:
  Request 'BJ2019FD7717' -> matched result: found=True
  Request 'BJ9999FD0000' -> matched result: found=False
  Request 'BJ2022FD6918' -> matched result: found=True

This worked because EVERY result carries its own originating input
along with it -- no separate bookkeeping needed by the calling code
to remember 'which result answers whi